## This notebook is trying to register a low res DESI and high RES Xenium OMI TIFF images together

In [2]:
import warnings
warnings.filterwarnings("ignore")

import spatialdata as sd
from spatialdata_io import xenium
from pathlib import Path
from spatialdata.models import Image2DModel
from spatialdata.transformations import Scale
import tifffile
import napari
import numpy as np
import pandas as pd

In [3]:
user_home = Path.home()

data_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" 
# Define the specific project folder
xenium_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "xenium"
msi_project_dir = user_home / "Desktop"/"multimodal_spatial_integration" / "data" / "msi"

raw_data_path = xenium_project_dir / "raw" / "output-XETG00169__0055588__55588_region_4__20250418__182706"
desi_img_data_path = msi_project_dir/ "raw"/ "3D_DESI_F5_Pos mode_40um_F5_5pos_40um_Features110425.ome.tif"
desi_flipped_img_data_path = msi_project_dir/ "processed"/ "3D_DESI_F5_Pos mode_40um_Flipped.ome.tif"

In [4]:
raw_data_path

PosixPath('/Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__55588_region_4__20250418__182706')

In [5]:
sdata = xenium(raw_data_path)

INFO     reading                                                                                                   
         /Users/lennonmccartney/Desktop/multimodal_spatial_integration/data/xenium/raw/output-XETG00169__0055588__5
         5588_region_4__20250418__182706/cell_feature_matrix.h5                                                    


In [6]:
sdata

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (5, 20495, 19973), (5, 10247, 9986), (5, 5123, 4993), (5, 2561, 2496), (5, 1280, 1248)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
│     └── 'nucleus_labels': DataTree[yx] (20495, 19973), (10247, 9986), (5123, 4993), (2561, 2496), (1280, 1248)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (52665, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (52665, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (48131, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (52665, 377)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points), cell_boundaries (Shapes), cell_circles (Shapes), nucleus_boundaries (Shapes)

In [7]:
desi_img = tifffile.imread(desi_img_data_path)

In [8]:
desi_img.shape

(88, 107, 102)

In [9]:
desi_img[87]

array([[ 59,  54,  22, ...,  12,  45,  74],
       [ 70, 180,   4, ...,   5,  86,  33],
       [ 82,  52, 142, ..., 437, 298,  20],
       ...,
       [ 11,  43, 140, ..., 245, 228, 475],
       [100,  32,  42, ..., 171,  92,  65],
       [105,  28,  75, ..., 208, 342, 356]],
      shape=(107, 102), dtype=uint16)

In [10]:
# print("\n\n=== Finding DESI Channels ===")
# print("We need to find channels for m/z: 616.25, 772.57, 893.75")
# print("\nLet's look at some channels to identify patterns...")

# # Quick visualization of several channels to find the right ones
# viewer = napari.Viewer()

# # Add a few DESI channels to explore
# # Start with some spread across the range
# test_channels = [3,17,23,25]

# for i in test_channels:
#     viewer.add_image(
#         desi_img[i], 
#         name=f"DESI_channel_{i}",
#         visible=(i == 0),  # Only first one visible by default
#         colormap='viridis'
#     )

# print(f"\nOpened napari with channels: {test_channels}")
# print("Toggle through channels to find ones with clear tissue structure")
# print("Note which channel numbers show the morphology you saw in QuPath")

# napari.run()

In [11]:
channel_indices = [3,17,23,25]
desi_selected = desi_img[channel_indices, :, :] 

In [12]:
# Create SpatialData image with proper coordinate system
desi_spatial = Image2DModel.parse(
    desi_selected,
    dims=("c", "y", "x"),
    transformations={
        "global": Scale(
            [40.0, 40.0],  # 40 μm per pixel
            axes=("y", "x")
        )
    },
    c_coords=["mz_204.13", "heme_B_616.25", "PE_PS_731.65", "PE_PS_734.60"]
)

In [13]:
# Add to spatialdata object
sdata.images["desi"] = desi_spatial

In [14]:
print("\nDESI image added to SpatialData object!")
print(f"Available images now: {list(sdata.images.keys())}")


DESI image added to SpatialData object!
Available images now: ['morphology_focus', 'desi']


In [15]:
sdata.images['desi']

<xarray.DataArray 'image' (c: 4, y: 107, x: 102)> Size: 87kB
dask.array<array, shape=(4, 107, 102), dtype=uint16, chunksize=(4, 107, 102), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U13 208B 'mz_204.13' 'heme_B_616.25' ... 'PE_PS_734.60'
  * y        (y) float64 856B 0.5 1.5 2.5 3.5 4.5 ... 103.5 104.5 105.5 106.5
  * x        (x) float64 816B 0.5 1.5 2.5 3.5 4.5 ... 97.5 98.5 99.5 100.5 101.5
Attributes:
    transform:  {'global': Scale (y, x)\n    [40. 40.]}

### Below is the commented code to attempt and write the spatial data object in a .zarr format for faster computation

In [16]:
# # Workaround: spatialdata 0.4.0 bug — dask's to_parquet() tries to JSON-serialize
# # the .attrs on the points DataFrame, which contains non-serializable Scale/Affine
# # transformation objects. Monkey-patch write_points to strip attrs only around the
# # to_parquet() call, so _get_transformations() still works normally.
# import spatialdata._io.io_points as _io_points
# _original_write_points = _io_points.write_points

# def _patched_write_points(points, group, name, group_type="ngff:points", format=_io_points.CurrentPointsFormat()):
#     saved = points.attrs.copy()
#     original_to_parquet = points.to_parquet

#     def safe_to_parquet(*args, **kwargs):
#         points.attrs.clear()
#         points.attrs["spatialdata_attrs"] = saved.get("spatialdata_attrs", {})
#         try:
#             return original_to_parquet(*args, **kwargs)
#         finally:
#             points.attrs.clear()
#             points.attrs.update(saved)

#     points.to_parquet = safe_to_parquet
#     try:
#         _original_write_points(points, group, name, group_type, format)
#     finally:
#         points.to_parquet = original_to_parquet

# _io_points.write_points = _patched_write_points

# output_zarr_path = project_dir / "processed" / "desi_xenium_register.zarr"
# sdata.write(output_zarr_path, overwrite=True)

### Using Napari-Spatial Data to visualize

In [17]:
from napari_spatialdata import Interactive

In [18]:
# interactive = Interactive(sdata)
# interactive.run()

In [19]:
from spatialdata.models import get_channel_names

In [20]:
# 1. Create Xenium reference - combine vessel + membrane markers
morph = sdata.images['morphology_focus']

channel_names = get_channel_names(morph)
channel_names

[np.str_('DAPI'),
 np.str_('ATP1A1/CD45/E-Cadherin'),
 np.str_('18S'),
 np.str_('AlphaSMA/Vimentin'),
 np.str_('dummy')]

In [21]:
# The full resolution data is typically at 'scale0'
morph_full = morph['scale0'].to_dataset()

In [22]:
print(f"Spatial dimensions: y={morph_full.dims['y']}, x={morph_full.dims['x']}")

Spatial dimensions: y=20495, x=19973


In [23]:
# The actual array is accessed via the data variable (usually called something like 'image' or the dataset name)
# Let's find the data variable name:
print(f"\nData variables: {list(morph_full.data_vars)}")


Data variables: ['image']


In [24]:
morph_array = morph_full['image']

# Extract relevant channels in scale0 which is the highest resolution we have
# Adjust indices based on actual channel order in your file
alphasma_vim = morph_array.sel(c='AlphaSMA/Vimentin').values  # Vessel marker
atp1a1 = morph_array.sel(c='ATP1A1/CD45/E-Cadherin').values  # Membrane marker
dapi = morph_array.sel(c='DAPI').values  # Nuclei marker

In [25]:
## similar to above extract the DESI channels as well
desi = sdata.images['desi']
desi_flipped = np.flip(desi,axis=2)
mz_204 = desi_flipped.sel(c='mz_204.13').values
heme_b = desi_flipped.sel(c='heme_B_616.25').values
pe_ps_731 = desi_flipped.sel(c='PE_PS_731.65').values
pe_ps_734 = desi_flipped.sel(c='PE_PS_734.60').values

In [26]:
## saving the flipped desi image
tifffile.imwrite(desi_flipped_img_data_path, desi_flipped)

In [27]:
# viewer = napari.Viewer()

# viewer.add_image(
#     atp1a1,
#     name='ATP1A1 (membranes)',
#     colormap='green',
#     scale=[0.2125, 0.2125],
#     blending='additive',
#     visible=True
# )

# viewer.add_image(
#     alphasma_vim,
#     name='AlphaSMA/Vimentin (vessels)',
#     colormap='magenta',
#     scale=[0.2125, 0.2125],
#     blending='additive',
#     visible=True
# )

# viewer.add_image(
#     dapi,
#     name='DAPI (nuclei)',
#     colormap='blue',
#     scale=[0.2125, 0.2125],  # Xenium pixel size
#     blending='additive',
#     visible=True
# )

# # DESI channels at 40 μm/pixel
# viewer.add_image(
#     mz_204,
#     name='DESI mz_204.13',
#     colormap='cyan',
#     scale=[40.0, 40.0],``
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     heme_b,
#     name='DESI Heme B (616.25)',
#     colormap='yellow',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     pe_ps_731,
#     name='DESI PE/PS (731.65)',
#     colormap='red',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

# viewer.add_image(
#     pe_ps_734,
#     name='DESI PE/PS (734.60)',
#     colormap='bop orange',
#     scale=[40.0, 40.0],
#     blending='additive',
#     visible=False
# )

In [28]:
# Normalize each channel to [0, 1]
def normalize(img):
    img = img.astype(float)
    return (img - img.min()) / (img.max() - img.min() + 1e-10)

In [29]:
# Xenium composite
xenium_composite = np.stack([
    normalize(alphasma_vim),  # Red - vessels
    normalize(atp1a1),        # Green - membranes
    np.zeros_like(atp1a1)     # Blue - empty
], axis=-1)

# DESI composite
desi_composite = np.stack([
    normalize(heme_b),      # Red - vessels (matches Xenium red)
    normalize(pe_ps_731), # Green - membranes (matches Xenium green)
    np.zeros_like(heme_b)   # Blue - empty
], axis=-1)

print("\n=== Composite Images Created ===")
print("Red channel: Vessels (AlphaSMA ↔ Heme B)")
print("Green channel: Membranes (ATP1A1 ↔ Lipid 731.65)")


=== Composite Images Created ===
Red channel: Vessels (AlphaSMA ↔ Heme B)
Green channel: Membranes (ATP1A1 ↔ Lipid 731.65)


In [30]:
# viewer = napari.Viewer()

# # Add Xenium composite (FIXED - reference)
# viewer.add_image(
#     xenium_composite,
#     name='Xenium_Composite (FIXED)',
#     rgb=True,
#     scale=[0.2125, 0.2125]
# )

# # Add DESI composite (MOVING - to be registered)
# viewer.add_image(
#     desi_composite,
#     name='DESI_Composite (MOVING)',
#     rgb=True,
#     scale=[40.0, 40.0],
#     opacity=0.7,
#     blending='additive'
# )

# # Optional: Add individual channels for reference
# viewer.add_image(
#     alphasma_vim,
#     name='Xenium_AlphaSMA (vessels only)',
#     colormap='red',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     atp1a1,
#     name='Xenium_ATP1A1 (membranes only)',
#     colormap='green',
#     scale=[0.2125, 0.2125],
#     visible=False,
#     blending='additive'
# )

# viewer.add_image(
#     heme_b,
#     name='DESI_HemeB (vessels only)',
#     colormap='red',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# viewer.add_image(
#     pe_ps_731,
#     name='DESI_Lipid731 (membranes only)',
#     colormap='green',
#     scale=[40.0, 40.0],
#     visible=False,
#     blending='additive',
#     opacity=0.7
# )

# # Add points layers for landmarks
# # border_width is relative (0-1) by default in this napari version
# fixed_points = viewer.add_points(
#     name='Xenium_landmarks (FIXED)',
#     ndim=2,
#     face_color='cyan',
#     size=200,
#     border_color='white',
#     border_width=0.2,
#     opacity=0.9
# )

# moving_points = viewer.add_points(
#     name='DESI_landmarks (MOVING)',
#     ndim=2,
#     face_color='yellow',
#     size=200,
#     border_color='white',
#     border_width=0.2,
#     opacity=0.9
# )

# napari.run()

In [31]:
moving_landmarks_path = data_dir /"3D_DESI_F5_Pos mode_40um_Flipped.ome.tif - Image0-points.tsv"
fixed_landmarks_path = data_dir /"morphology_focus_0002.ome.tif-points.tsv"

In [32]:
xenium_df = pd.read_csv(fixed_landmarks_path, sep='\t')
desi_df = pd.read_csv(moving_landmarks_path, sep='\t')

In [33]:
# Extract point number from the class column (last part after '_')
xenium_df['point_num'] = xenium_df['class'].str.extract(r'_(\d+)$').astype(int)
desi_df['point_num'] = desi_df['class'].str.extract(r'_(\d+)$').astype(int)

# Add coordinates in microns
xenium_df['x_microns'] = xenium_df['x'] * 0.2125
xenium_df['y_microns'] = xenium_df['y'] * 0.2125
desi_df['x_microns'] = desi_df['x'] * 40.0
desi_df['y_microns'] = desi_df['y'] * 40.0

# Join on point number
landmarks_df = xenium_df[['point_num', 'x', 'y', 'x_microns', 'y_microns']].merge(
    desi_df[['point_num', 'x', 'y', 'x_microns', 'y_microns']],
    on='point_num',
    suffixes=('_xenium', '_desi')
).sort_values('point_num').reset_index(drop=True)

landmarks_df

,point_num,x_xenium,y_xenium,x_microns_xenium,y_microns_xenium,x_desi,y_desi,x_microns_desi,y_microns_desi
0,1,402.770996,457.542877,85.588837,97.227861,6.131444,1.119414,245.257771,44.776576
1,2,3699.857910,3422.365234,786.219806,727.252612,22.170194,16.869177,886.807741,674.767087
2,3,16325.912109,15051.625000,3469.256323,3198.470313,92.971879,80.301708,3718.875175,3212.068319
3,4,15124.647461,11907.891602,3213.987585,2530.426965,86.180697,63.829479,3447.227891,2553.179161
4,5,8377.121094,5697.099609,1780.138232,1210.633667,45.722591,30.740528,1828.903642,1229.621115
5,6,8351.562045,13748.125971,1774.706935,2921.476769,52.224787,75.677924,2088.991468,3027.116977


In [34]:
import SimpleITK as sitk

# Convert to SimpleITK images (use AlphaSMA as the Xenium reference, Heme B as DESI reference)
xenium_sitk = sitk.GetImageFromArray(alphasma_vim)
xenium_sitk.SetSpacing([0.2125, 0.2125])  # x, y spacing in microns
xenium_sitk.SetOrigin([0.0, 0.0])

desi_sitk = sitk.GetImageFromArray(heme_b)
desi_sitk.SetSpacing([40.0, 40.0])  # x, y spacing in microns
desi_sitk.SetOrigin([0.0, 0.0])

print(f"Xenium SimpleITK: size={xenium_sitk.GetSize()}, spacing={xenium_sitk.GetSpacing()}")
print(f"DESI SimpleITK: size={desi_sitk.GetSize()}, spacing={desi_sitk.GetSpacing()}")

# Prepare landmarks for SimpleITK — list of [x, y] in physical coordinates (microns)
fixed_landmarks = [[float(row.x_microns_xenium), float(row.y_microns_xenium)] for _, row in landmarks_df.iterrows()]
moving_landmarks = [[float(row.x_microns_desi), float(row.y_microns_desi)] for _, row in landmarks_df.iterrows()]

print(f"\nFixed landmarks (Xenium, microns): {fixed_landmarks}")
print(f"Moving landmarks (DESI, microns): {moving_landmarks}")

Xenium SimpleITK: size=(19973, 20495), spacing=(0.2125, 0.2125)
DESI SimpleITK: size=(102, 107), spacing=(40.0, 40.0)

Fixed landmarks (Xenium, microns): [[85.58883666992188, 97.22786140441895], [786.219805908203, 727.2526123046874], [3469.2563232421876, 3198.4703125], [3213.987585449219, 2530.426965332031], [1780.138232421875, 1210.6336669921875], [1774.7069346144187, 2921.476768914608]]
Moving landmarks (DESI, microns): [[245.25777082429371, 44.776576312963456], [886.8077406786348, 674.7670872510103], [3718.875175170773, 3212.0683193776395], [3447.227890637853, 2553.179161148856], [1828.9036423566315, 1229.6211152331432], [2088.9914679732565, 3027.1169767169285]]
